# CA Exam Checker AI — Evaluation Supervisor Fine-Tuning

This notebook trains a **separate research adapter** for auditing structured evaluation output. It does not change the existing study-planner model or the website's deployed evaluator.

**Safety notes**
- The starter dataset is synthetic only; no student copies or ICAI solution text are included.
- A synthetic test score is only a pipeline smoke test, not proof that marks are academically correct.
- Do not promote this adapter to production until it passes a separate, human-reviewed and legally permitted benchmark.
- The adapter can flag issues; the TypeScript integrity gates remain authoritative for exact checks.


In [ ]:
import os, sys, subprocess, json, torch
from pathlib import Path

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU runtime required. In Colab choose Runtime > Change runtime type > T4 GPU, then rerun this cell."
    )
print("✅ GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

from google.colab import drive
drive.mount("/content/drive")


In [ ]:
REPO_DIR = "/content/AI-Exam"
BRANCH = "main"
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         "https://github.com/thomasshelby-pixel/AI-Exam.git", REPO_DIR],
        check=True
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)

PROJECT_DIR = f"{REPO_DIR}/research/ca-evaluation-supervisor-ai"
%cd /content/AI-Exam/research/ca-evaluation-supervisor-ai
%pip install -q -r requirements-colab.txt
print("✅ Training sources ready:", PROJECT_DIR)


In [ ]:
# Generate reproducible synthetic examples for train / validation / held-out test.
# The generator does not read Google Drive, student uploads, or ICAI PDFs.
subprocess.run(
    [sys.executable, "generate_supervisor_dataset.py",
     "--output-dir", "/content/ca-evaluation-supervisor-data",
     "--train-per-category", "160",
     "--validation-per-category", "30",
     "--test-per-category", "40",
     "--seed", "20261017"],
    check=True
)
manifest = json.loads(Path("/content/ca-evaluation-supervisor-data/dataset_manifest.json").read_text())
print(json.dumps(manifest, indent=2))
assert manifest["dataset_kind"] == "synthetic_only"
assert manifest["counts"]["train"]["examples"] > manifest["counts"]["test"]["examples"]


## Train an isolated QLoRA adapter

The trainer deliberately refuses CPU-only runtime and refuses output locations that could overwrite the old planner model. Default training uses a T4 GPU, 4-bit QLoRA, and a new versioned Drive output folder.

Do not rerun the training cell while it is running. A completed run should save adapter weights plus a training manifest.


In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluation_supervisor_smollm2_360m_v1_20261017"
subprocess.run(
    [sys.executable, "train_supervisor_qlora.py",
     "--train-file", "/content/ca-evaluation-supervisor-data/train.jsonl",
     "--validation-file", "/content/ca-evaluation-supervisor-data/validation.jsonl",
     "--output-dir", OUTPUT_DIR,
     "--base-model", "HuggingFaceTB/SmolLM2-360M-Instruct",
     "--epochs", "3",
     "--batch-size", "2",
     "--grad-accum", "8",
     "--learning-rate", "0.0001",
     "--max-seq-length", "1024"],
    check=True
)
print("✅ Training process exited successfully:", OUTPUT_DIR)


## Evaluate on held-out synthetic cases

This runs the fine-tuned adapter on held-out examples and (for comparison) the untouched base model. The report includes JSON parse rate, decision accuracy, issue-code precision/recall/F1, exact issue-set accuracy, and the rate at which flagged synthetic cases were incorrectly passed.

**Do not describe these synthetic metrics as ICAI grading accuracy.**


In [ ]:
EVAL_DIR = "/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluation_supervisor_evaluation_v1_20261017"
subprocess.run(
    [sys.executable, "evaluate_supervisor.py",
     "--test-file", "/content/ca-evaluation-supervisor-data/test.jsonl",
     "--adapter-dir", OUTPUT_DIR,
     "--base-model", "HuggingFaceTB/SmolLM2-360M-Instruct",
     "--output-dir", EVAL_DIR,
     "--limit", "120",
     "--compare-base"],
    check=True
)
metrics_path = Path(EVAL_DIR) / "metrics.json"
metrics = json.loads(metrics_path.read_text())
print(json.dumps(metrics, indent=2))


In [ ]:
# Package only the root adapter/tokenizer files and evaluation evidence.
# Trainer checkpoint directories are intentionally excluded from the deployment package.
import shutil, zipfile

EXPORT_DIR = "/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluation_supervisor_smollm2_360m_v1_20261017_EXPORT"
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
adapter_root = Path(OUTPUT_DIR)
for file_path in adapter_root.iterdir():
    if file_path.is_file():
        shutil.copy2(file_path, Path(EXPORT_DIR) / file_path.name)
for name in ["metrics.json", "base_metrics.json", "predictions.jsonl", "base_predictions.jsonl"]:
    source = Path(EVAL_DIR) / name
    if source.is_file():
        shutil.copy2(source, Path(EXPORT_DIR) / name)

package_path = "/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluation_supervisor_adapter_package_20261017.zip"
with zipfile.ZipFile(package_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in Path(EXPORT_DIR).iterdir():
        if file_path.is_file():
            archive.write(file_path, arcname=file_path.name)

print("✅ Adapter package:", package_path)
print("✅ Evaluation evidence:", str(Path(EXPORT_DIR) / "metrics.json"))
print("⚠️ This is a research artifact; it is not wired into the live website yet.")


## Deployment gate — intentionally not automatic

The model has **not** been deployed by running this notebook. Next, review:
1. base vs fine-tuned synthetic metrics;
2. whether the adapter returns valid JSON and avoids false PASS decisions;
3. a separate benchmark of human-reviewed, legally permitted real evaluation examples.

Only then should we build an inference endpoint and connect it to the website. The model must never override the deterministic integrity gates or approve a copy when references, answer mapping, or handwriting are unresolved.
